# ECU states from a recording

Can a recording's own signals say what the engine will do half a second from now?
This notebook asks it of one state, **fuel cut**: the injectors off and the exhaust
reading lean, on the rev limiter, on the launch two-step, or with the throttle lifted at
speed. It reads a recording through the `havne`
package's reader (ADR-0049), turns it into 10 Hz columns, labels each column with the
cut state 0.5 s later, and fits two small models against a baseline that only says
"the state stays as it is".

The data is `data/drive-loop/`: one loop of the mock transport's synthetic drive cycle,
key-on to the engine's stop, made by `data/make_drive_loop.py`. It holds four signals:
RPM, vehicle speed, throttle position and lambda. The same code reads any recording with
those signals, a real car's included.

Needs numpy and scikit-learn beside the `havne` package.

In [1]:
import os

import numpy as np

import havne

FOLDER = os.path.join("data", "drive-loop")
NAMES = ["RPM", "Vehicle Speed", "Throttle Position", "Lambda"]

rec = havne.Recording(FOLDER)
for s in rec.signals:
    print(f"{s.name:18} {s.samples:6} samples, {s.lowest:g} to {s.highest:g} {s.unit}")
print(f"{rec.info.duration_us / 1e6:.1f} s")

RPM                  2879 samples, 0 to 7064.75 RPM
Vehicle Speed        2879 samples, 0 to 173 km/h
Throttle Position    2879 samples, 14.5098 to 85.8824 %
Lambda               2879 samples, 0.851471 to 1.98999 λ
153.6 s


## Columns at 10 Hz

`Recording.query` cuts a time span into columns and gives each signal's lowest and
highest value in each, the way the log viewer draws a chart. A column's value here is
its midpoint; a column with no sample keeps the one before.

In [2]:
STEP_US = 100_000

t0 = min(rec.signal(n).first_us for n in NAMES)
t1 = max(rec.signal(n).last_us for n in NAMES) + 1
n = (t1 - t0) // STEP_US
columns = rec.query(NAMES, t0, t0 + n * STEP_US, n)


def series(cols, pick):
    out = np.full(len(cols), np.nan)
    for i, c in enumerate(cols):
        if c.samples:
            out[i] = pick(c)
    for i in range(1, len(out)):          # carry the last value across an empty column
        if np.isnan(out[i]):
            out[i] = out[i - 1]
    return out


mid = {name: series(cols, lambda c: (c.lowest + c.highest) / 2)
       for name, cols in zip(NAMES, columns)}
low = {name: series(cols, lambda c: c.lowest) for name, cols in zip(NAMES, columns)}
t = np.arange(n) * STEP_US / 1e6
print(f"{n} columns of {STEP_US // 1000} ms")

1534 columns of 100 ms


## The state and the label

A column is in fuel cut when its lowest lambda is over 1.3 (lean past anything the
engine runs on fuel) while its lowest RPM is over 500 (the engine turning, not stopped).
The label is that state `AHEAD` columns later: 0.5 s.

In [3]:
AHEAD = 5

cut = (low["Lambda"] > 1.3) & (low["RPM"] > 500)
starts = np.flatnonzero(cut[1:] & ~cut[:-1]) + 1
print(f"in fuel cut {cut.mean():.1%} of the time, {len(starts)} times, "
      f"first at {t[starts[0]]:.1f} s" if len(starts) else "never in fuel cut")

in fuel cut 19.0% of the time, 37 times, first at 40.9 s


## Features

What a column knows now: RPM, speed and throttle, how much each moved over the last
0.5 s, and how long the throttle has been closed (under 16 %, the mock's closed
throttle reads about 14.5 %). Lambda stays out: it is the state itself.

In [4]:
CLOSED = 16.0
BACK = 5

rpm, speed, throttle = mid["RPM"], mid["Vehicle Speed"], mid["Throttle Position"]
closed_for = np.zeros(n)
for i in range(1, n):
    closed_for[i] = closed_for[i - 1] + STEP_US / 1e6 if throttle[i] < CLOSED else 0.0


def slope(x):
    d = np.zeros_like(x)
    d[BACK:] = x[BACK:] - x[:-BACK]
    return d


FEATURES = ["rpm", "speed", "throttle", "rpm change", "speed change", "throttle change",
            "closed for (s)"]
X_all = np.column_stack([rpm, speed, throttle, slope(rpm), slope(speed), slope(throttle),
                         closed_for])
rows = np.arange(BACK, n - AHEAD)
X, y, now = X_all[rows], cut[rows + AHEAD], cut[rows]
changes = y != now
print(f"{len(rows)} rows, {y.mean():.1%} of them labeled fuel cut, "
      f"{changes.sum()} where the state changes within 0.5 s")

1524 rows, 19.2% of them labeled fuel cut, 90 where the state changes within 0.5 s


## Models, judged on held-out time

Five blocked folds: each fold is one stretch of the loop, held out while the models
learn from the rest, so a model never sees the moments next to the ones it is judged on.
The baseline predicts the state now. It is right most of the time and wrong at every
change, so the changes are where a model has to earn its place.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

folds = KFold(n_splits=5, shuffle=False)
models = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "decision tree": DecisionTreeClassifier(max_depth=3, random_state=0),
}
predicted = {"baseline (the state now)": now}
for name, model in models.items():
    predicted[name] = cross_val_predict(model, X, y, cv=folds)


def scores(p):
    hit = (p & y).sum()
    precision = hit / p.sum() if p.sum() else 0.0
    recall = hit / y.sum() if y.sum() else 0.0
    return precision, recall, (p == y)[changes].mean()


print(f"{'':26} precision  recall  right at the changes")
for name, p in predicted.items():
    precision, recall, at_changes = scores(p.astype(bool))
    print(f"{name:26} {precision:9.2f} {recall:7.2f} {at_changes:21.2f}")

                           precision  recall  right at the changes
baseline (the state now)        0.85    0.85                  0.00
logistic regression             0.84    0.76                  0.56
decision tree                   0.98    0.76                  0.61


## What the tree learned

Fitted on the whole loop, the tree's rules can be read off and set beside the drive
cycle's own (`core/src/mock_drive_cycle.h`), which cuts fuel on the rev limiter, on the
launch two-step, and once the throttle has been closed for half a second above 1400 RPM
while the car is moving.

In [6]:
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)
print(export_text(tree, feature_names=FEATURES, decimals=1))

|--- rpm change <= -34.8
|   |--- speed <= 49.8
|   |   |--- rpm <= 2464.2
|   |   |   |--- class: False
|   |   |--- rpm >  2464.2
|   |   |   |--- class: True
|   |--- speed >  49.8
|   |   |--- throttle <= 14.6
|   |   |   |--- class: True
|   |   |--- throttle >  14.6
|   |   |   |--- class: False
|--- rpm change >  -34.8
|   |--- rpm <= 6865.2
|   |   |--- throttle change <= -60.7
|   |   |   |--- class: True
|   |   |--- throttle change >  -60.7
|   |   |   |--- class: False
|   |--- rpm >  6865.2
|   |   |--- closed for (s) <= 0.1
|   |   |   |--- class: False
|   |   |--- closed for (s) >  0.1
|   |   |   |--- class: True



In [7]:
# The notebook's own check, run with it (ExampleTest.notebook): the tree must
# be right at more than half the changes, where the baseline is right at none,
# and rarely call a cut that doesn't come.
precision, _, at_changes = scores(predicted["decision tree"].astype(bool))
assert at_changes > 0.5 and precision > 0.9, (precision, at_changes)
rec.close()